# 9 — Stage 1 Results (E1 + E2)

**E1 — Modularity:** Do universal circuits survive marginalization?
Measured by circuit size (number of neurons that survive intersection across all pairings).

**E2 — Concept vs Token decomposition:** What fraction of a concept's circuit
is concept-only (not shared with the token checker)?

Both results are derived from the neuron list CSVs produced by notebook 4/R4.
Covers all available language x model x threshold combinations.

In [ ]:
# Cell 1 – Configuration
import subprocess, sys, os
for pkg in ["numpy", "pandas", "matplotlib", "seaborn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

MODELS = ["QW"]  # add "DS" when available
LANGS = ["P", "R"]
LANG_LABELS = {"P": "Python", "R": "Rust"}
MODEL_LABELS = {"QW": "Qwen", "DS": "DeepSeek"}

EPSILONS = ["0.001", "0.1", "0.5"]
CONSISTENCIES = ["0.2", "0.5", "0.8"]
N_LAYERS = 28

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import shutil
    mp = "/content/drive"
    subprocess.run(["fusermount", "-uz", mp], capture_output=True)
    if os.path.isdir(mp):
        shutil.rmtree(mp, ignore_errors=True)
    drive.mount(mp)
    DATA_DIR = Path("/content/drive/MyDrive/DATA/New-Atlas")
else:
    DATA_DIR = Path("/Users/piotrwilam/Data/New-Atlas")

print(f"Data: {DATA_DIR}")

In [ ]:
# Cell 2 – Classification functions & load data

PYTHON_MODULAR = {"ast__Import", "ast__Break", "ast__Pass",
                  "ast__ImportFrom", "ast__Continue", "ast__Assert"}
RUST_MODULAR = {"rust__Use", "rust__Mod", "rust__Break", "rust__Continue",
                "rust__Return", "rust__Unsafe", "rust__Await"}
RUST_CONSTRUCTS = {
    "rust__For", "rust__While", "rust__Loop", "rust__If", "rust__Match",
    "rust__Fn", "rust__Closure", "rust__Let", "rust__LetMut", "rust__Const",
    "rust__Static", "rust__Struct", "rust__Enum", "rust__TypeAlias",
    "rust__Impl", "rust__Trait", "rust__Use", "rust__Mod",
    "rust__Return", "rust__Break", "rust__Continue",
    "rust__Async", "rust__Await", "rust__Unsafe",
    "rust__Ref", "rust__RefMut", "rust__Deref", "rust__Lifetime",
    "rust__Macro", "rust__Attribute", "rust__QuestionMark",
}

def classify(obj, lang):
    if lang == "P":
        if obj.startswith("builtin__"):
            return "Builtin"
        if obj in PYTHON_MODULAR:
            return "Modular"
        return "Non-modular"
    else:
        if obj in RUST_MODULAR:
            return "Modular"
        if obj in RUST_CONSTRUCTS:
            return "Non-modular"
        return "Object"


frames = {}  # (lang, model, eps, cons) -> DataFrame

for lang in LANGS:
    for model in MODELS:
        prefix = f"{lang}_{model}_"
        for eps in EPSILONS:
            for cons in CONSISTENCIES:
                fname = f"{prefix}4_neuron_list_eps{eps}_cons{cons}_all_layers_both.csv"
                path = DATA_DIR / fname
                if not path.exists():
                    continue
                df = pd.read_csv(path)
                df["circuit_size"] = df["n_concept_only"] + df["n_both"]
                df["concept_fraction"] = (df["n_concept_only"] / df["circuit_size"]).replace([np.inf], 0).fillna(0)
                df["group"] = df["object"].apply(lambda o: classify(o, lang))
                frames[(lang, model, eps, cons)] = df

print(f"Loaded {len(frames)} neuron list files")
for key in sorted(frames.keys()):
    print(f"  {key[0]}_{key[1]} eps={key[2]} cons={key[3]}: {len(frames[key])} rows")

In [ ]:
# Cell 3 – E1: Modularity — Circuit survival after marginalization
#
# A concept's universal circuit is the intersection of its pair circuits
# across all pairings. If the circuit survives (size > 0), the concept
# has a modular representation. Circuit size = |concept_only| + |both|.

print("E1: MODULARITY — Circuit survival after marginalization")
print("=" * 70)

for lang in LANGS:
    for model in MODELS:
        label = f"{LANG_LABELS[lang]} / {MODEL_LABELS[model]}"

        # Use a representative setting
        df = frames.get((lang, model, "0.5", "0.8"))
        if df is None:
            print(f"\n--- {label}: missing (eps=0.5, cons=0.8) ---")
            continue

        print(f"\n--- {label} (eps=0.5, cons=0.8) ---")

        # Per-object: mean circuit size across layers
        obj_summary = (df.groupby(["object", "group"])
                       .agg(mean_circuit_size=("circuit_size", "mean"),
                            max_circuit_size=("circuit_size", "max"),
                            nonzero_layers=("circuit_size", lambda x: (x > 0).sum()))
                       .reset_index()
                       .sort_values("mean_circuit_size", ascending=False))

        # Survival rate: fraction of objects with circuit_size > 0 at any layer
        n_objects = obj_summary["object"].nunique()
        n_surviving = (obj_summary["max_circuit_size"] > 0).sum()
        print(f"  Objects with surviving circuit: {n_surviving}/{n_objects} ({100*n_surviving/n_objects:.0f}%)")

        # Per-group summary
        for group in sorted(obj_summary["group"].unique()):
            gsub = obj_summary[obj_summary["group"] == group]
            n = len(gsub)
            surv = (gsub["max_circuit_size"] > 0).sum()
            mean_sz = gsub["mean_circuit_size"].mean()
            mean_layers = gsub["nonzero_layers"].mean()
            print(f"  {group:15s}  objects={n:3d}  surviving={surv:3d} ({100*surv/n:.0f}%)  "
                  f"mean_size={mean_sz:.1f}  mean_active_layers={mean_layers:.1f}")

        # Top 10 by circuit size
        print(f"\n  Top 10 by mean circuit size:")
        for _, r in obj_summary.head(10).iterrows():
            print(f"    {r['object']:30s} [{r['group']:15s}]  mean={r['mean_circuit_size']:.1f}  "
                  f"max={r['max_circuit_size']}  active_layers={r['nonzero_layers']}")

In [ ]:
# Cell 4 – E1: Circuit size across threshold grid

print("E1: Mean circuit size across parameter grid")
print("=" * 70)

for lang in LANGS:
    for model in MODELS:
        label = f"{LANG_LABELS[lang]} / {MODEL_LABELS[model]}"
        print(f"\n--- {label} ---")

        rows = []
        for eps in EPSILONS:
            for cons in CONSISTENCIES:
                df = frames.get((lang, model, eps, cons))
                if df is None:
                    rows.append({"eps": eps, "cons": cons, "Mean circuit": "-", "Surviving %": "-"})
                    continue

                obj_max = df.groupby("object")["circuit_size"].max()
                n_surv = (obj_max > 0).sum()
                n_total = len(obj_max)
                mean_cs = df["circuit_size"].mean()

                rows.append({
                    "eps": eps, "cons": cons,
                    "Mean circuit": f"{mean_cs:.1f}",
                    "Surviving %": f"{100*n_surv/n_total:.0f}%",
                })

        display(pd.DataFrame(rows))

In [ ]:
# Cell 5 – E1: Circuit size layer profile

print("E1: Circuit size by layer")
print("=" * 70)

fig, axes = plt.subplots(1, len(LANGS), figsize=(7*len(LANGS), 5), sharey=False)
if len(LANGS) == 1:
    axes = [axes]

for ax, lang in zip(axes, LANGS):
    for model in MODELS:
        df = frames.get((lang, model, "0.5", "0.8"))
        if df is None:
            continue

        for group in sorted(df["group"].unique()):
            gsub = df[df["group"] == group]
            layer_means = gsub.groupby("layer")["circuit_size"].mean()
            ax.plot(layer_means.index, layer_means.values, label=group, linewidth=1.5)

    ax.set_xlabel("Layer")
    ax.set_ylabel("Mean circuit size")
    ax.set_title(f"{LANG_LABELS[lang]} (eps=0.5, cons=0.8)")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(DATA_DIR / "9_E1_circuit_size_profile.png", dpi=150)
plt.show()

In [ ]:
# Cell 6 – E2: Concept fraction grid (Table 1)

print("E2: Concept fraction across parameter grid")
print("=" * 70)

for lang in LANGS:
    for model in MODELS:
        label = f"{LANG_LABELS[lang]} / {MODEL_LABELS[model]}"
        print(f"\n--- {label} ---")

        rows = []
        for eps in EPSILONS:
            for cons in CONSISTENCIES:
                df = frames.get((lang, model, eps, cons))
                if df is None:
                    rows.append({"eps": eps, "cons": cons,
                                 "Construct CF": "-", "Object CF": "-"})
                    continue

                concept_df = df[df["group"].isin(["Modular", "Non-modular"])]
                object_df = df[df["group"].isin(["Builtin", "Object"])]

                c_cf = (concept_df["n_concept_only"].sum() / concept_df["circuit_size"].sum()
                        if concept_df["circuit_size"].sum() > 0 else 0)
                o_cf = (object_df["n_concept_only"].sum() / object_df["circuit_size"].sum()
                        if object_df["circuit_size"].sum() > 0 else 0)

                rows.append({
                    "eps": eps, "cons": cons,
                    "Construct CF": f"{c_cf:.1%}",
                    "Object CF": f"{o_cf:.1%}",
                })

        display(pd.DataFrame(rows))

In [ ]:
# Cell 7 – E2: Neuron counts at mid-layer (Table 2)

MID_LAYER = 14
print(f"E2: Neuron counts at layer {MID_LAYER} (eps=0.5, cons=0.8)")
print("=" * 70)

for lang in LANGS:
    for model in MODELS:
        label = f"{LANG_LABELS[lang]} / {MODEL_LABELS[model]}"
        df = frames.get((lang, model, "0.5", "0.8"))
        if df is None:
            print(f"\n--- {label}: missing ---")
            continue

        l_mid = df[df["layer"] == MID_LAYER]
        rows = []
        for group_name in sorted(l_mid["group"].unique()):
            g = l_mid[l_mid["group"] == group_name]
            c = g["n_concept_only"].sum()
            s = g["n_both"].sum()
            t = g["n_token_only"].sum()
            sz = c + s
            cf = c / sz if sz > 0 else 0
            rows.append({"Group": group_name, "Concept-only": c, "Shared": s,
                         "Token-only": t, "Circuit size": sz,
                         "Concept fraction": f"{cf:.1%}"})

        print(f"\n--- {label} ---")
        display(pd.DataFrame(rows))

In [ ]:
# Cell 8 – E2: Layer-by-layer concept fraction (Table 3)

print("E2: Layer-by-layer concept fraction (eps=0.001, cons=0.8)")
print("=" * 70)

for lang in LANGS:
    for model in MODELS:
        label = f"{LANG_LABELS[lang]} / {MODEL_LABELS[model]}"
        df = frames.get((lang, model, "0.001", "0.8"))
        if df is None:
            print(f"\n--- {label}: missing ---")
            continue

        available_layers = sorted(df["layer"].unique())
        rows = []
        for group_label in sorted(df["group"].unique()):
            row = {"Group": group_label}
            for layer in available_layers:
                sub = df[(df["layer"] == layer) & (df["group"] == group_label)]
                c = sub["n_concept_only"].sum()
                sz = sub["circuit_size"].sum()
                cf = c / sz if sz > 0 else 0
                row[f"L{layer}"] = f"{cf:.1%}"
            rows.append(row)

        print(f"\n--- {label} ---")
        display(pd.DataFrame(rows))

In [ ]:
# Cell 9 – E2: Concept fraction layer profile (figure)

fig, axes = plt.subplots(1, len(LANGS), figsize=(7*len(LANGS), 5), sharey=False)
if len(LANGS) == 1:
    axes = [axes]

for ax, lang in zip(axes, LANGS):
    for model in MODELS:
        df = frames.get((lang, model, "0.001", "0.8"))
        if df is None:
            continue

        for group in sorted(df["group"].unique()):
            gsub = df[df["group"] == group]
            layer_cf = []
            for lid in range(N_LAYERS):
                lsub = gsub[gsub["layer"] == lid]
                c = lsub["n_concept_only"].sum()
                sz = lsub["circuit_size"].sum()
                layer_cf.append(c / sz if sz > 0 else 0)
            ax.plot(range(N_LAYERS), layer_cf, label=group, linewidth=1.5)

    ax.set_xlabel("Layer")
    ax.set_ylabel("Concept fraction")
    ax.set_title(f"{LANG_LABELS[lang]} (eps=0.001, cons=0.8)")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(DATA_DIR / "9_E2_concept_fraction_profile.png", dpi=150)
plt.show()

In [ ]:
# Cell 10 – Per-object detail (eps=0.5, cons=0.8)

print("PER-OBJECT DETAIL (eps=0.5, cons=0.8)")
print("=" * 70)

for lang in LANGS:
    for model in MODELS:
        label = f"{LANG_LABELS[lang]} / {MODEL_LABELS[model]}"
        df = frames.get((lang, model, "0.5", "0.8"))
        if df is None:
            continue

        summary = (df.groupby(["object", "group"])
                   .agg(mean_concept_only=("n_concept_only", "mean"),
                        mean_both=("n_both", "mean"),
                        mean_circuit=("circuit_size", "mean"),
                        mean_cf=("concept_fraction", "mean"))
                   .reset_index()
                   .sort_values("mean_cf", ascending=False))

        print(f"\n--- {label} ---")
        display(summary.head(30))

In [ ]:
# Cell 11 – Save

# Save per-combo summaries
for (lang, model, eps, cons), df in frames.items():
    summary = (df.groupby(["object", "group"])
               .agg(mean_concept_only=("n_concept_only", "mean"),
                    mean_both=("n_both", "mean"),
                    mean_circuit=("circuit_size", "mean"),
                    mean_cf=("concept_fraction", "mean"))
               .reset_index())
    fname = f"9_results_{lang}_{model}_eps{eps}_cons{cons}.csv"
    summary.to_csv(DATA_DIR / fname, index=False)

print(f"Saved {len(frames)} summary CSVs")
print("\n9_results_stage1 complete.")